In [5]:
import pandas as pd
import numpy as np
from datetime import datetime

# 1. Configuration (Absolute paths pointing to Part 2 data directory)
COUNTRIES = ["US", "GB", "IN"]
FILE_PATHS = {
    "US": "/data/USvideos.csv",
    "GB": "/data/GBvideos.csv",
    "IN": "/data/INvideos.csv",
    "categories": "/data/categories.csv"
}

# Execution Header
print("==================================================")
print("=== TrendPulse Part 2: NumPy & Pandas Wrangling ===")
print(f"Target Countries : {', '.join(COUNTRIES)}")
print(f"Execution Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("==================================================")

=== TrendPulse Part 2: NumPy & Pandas Wrangling ===
Target Countries : US, GB, IN
Execution Started: 2026-11-09 13:53:06


In [6]:
# Load per-country dataframes and combine
dfs = []
for code in COUNTRIES:
    df_temp = pd.read_csv(FILE_PATHS[code])
    df_temp['country'] = code
    dfs.append(df_temp)

df_raw = pd.concat(dfs, ignore_index=True)

# Load categories lookup and merge (Left merge to keep all trending records)
categories_df = pd.read_csv(FILE_PATHS["categories"])
categories_df['category_id'] = categories_df['category_id'].astype(str)
df_raw['category_id'] = df_raw['category_id'].astype(str)

df = pd.merge(
    df_raw,
    categories_df[['category_id', 'category_name']],
    on='category_id',
    how='left'
)

print(f"Combined Dataset Shape: {df.shape}")
print("\n--- Columns & Data Types ---")
df.info()

Combined Dataset Shape: (46230, 18)

--- Columns & Data Types ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46230 entries, 0 to 46229
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   video_id                46230 non-null  object 
 1   trending_date           46230 non-null  object 
 2   title                   46230 non-null  object 
 3   channel_title           46230 non-null  object 
 4   category_id             46230 non-null  object 
 5   publish_time            46230 non-null  object 
 6   tags                    46230 non-null  object 
 7   views                   46230 non-null  int64  
 8   likes                   46218 non-null  float64
 9   dislikes                46230 non-null  int64  
 10  comment_count           46194 non-null  float64
 11  thumbnail_link          46230 non-null  object 
 12  comments_disabled       46230 non-null  bool   
 13  ratings_disabled        4

In [7]:
import numpy as np

# 1. Extract columns into 1D NumPy arrays
views = df["views"].to_numpy(dtype=float)
likes = df["likes"].fillna(0).to_numpy(dtype=float)
dislikes = df["dislikes"].to_numpy(dtype=float)
comment_count = df["comment_count"].fillna(0).to_numpy(dtype=float)

print(f"Views Array Shape: {views.shape}, dtype: {views.dtype}")

# 2. Vectorized Safe Ratios (Handling zero denominators)
like_ratio = np.where((likes + dislikes) > 0, likes / (likes + dislikes), 0.0)
comment_rate = np.where(views > 0, comment_count / views, 0.0)

# 3. Min-Max Normalization helper function
def min_max_scale(arr):
    min_val, max_val = np.min(arr), np.max(arr)
    if max_val - min_val == 0:
        return np.zeros_like(arr)
    return (arr - min_val) / (max_val - min_val)

norm_views = min_max_scale(views)
norm_likes = min_max_scale(likes)
norm_comments = min_max_scale(comment_count)

# 4. Weighted Engagement Score (40% Views, 40% Likes, 20% Comments)
df["engagement_score"] = (0.40 * norm_views) + (0.40 * norm_likes) + (0.20 * norm_comments)

# 5. Log-scaled view count
df["log_views"] = np.log1p(views)

# Flag Top 10% engagement score using boolean masks
threshold_90th = np.percentile(df["engagement_score"], 90)
df["top_10pct_engagement"] = df["engagement_score"] >= threshold_90th

print(f"90th Percentile Engagement Score Threshold: {threshold_90th:.6f}")
print(f"Records in Top 10%: {df['top_10pct_engagement'].sum():,}")

Views Array Shape: (46230,), dtype: float64
90th Percentile Engagement Score Threshold: 0.005103
Records in Top 10%: 4,623


/tmp/ipykernel_10050/1029724440.py:12: RuntimeWarning: invalid value encountered in divide
  like_ratio = np.where((likes + dislikes) > 0, likes / (likes + dislikes), 0.0)


In [8]:
# Report missing values
missing_summary = df.isna().sum()
missing_pct = (df.isna().sum() / len(df)) * 100
missing_df = pd.DataFrame({"Missing Count": missing_summary, "Percentage (%)": missing_pct})
print("--- Missing Values Summary ---")
print(missing_df[missing_df["Missing Count"] > 0])

# Strategy: Fill missing descriptions with empty string
df["description"] = df["description"].fillna("")

# Define duplicates: Same video_id appearing on the same trending_date in the same country
initial_count = len(df)
df = df.drop_duplicates(subset=["video_id", "trending_date", "country"])
removed_duplicates = initial_count - len(df)
print(f"\nDuplicates removed (video_id + trending_date + country): {removed_duplicates:,}")

--- Missing Values Summary ---
               Missing Count  Percentage (%)
likes                     12        0.025957
comment_count             36        0.077872
description             1624        3.512870

Duplicates removed (video_id + trending_date + country): 230


In [9]:
# (a) Videos with >= 1M views and > 50k likes
high_perf = df.loc[(df["views"] >= 1_000_000) & (df["likes"] > 50_000), ["title", "channel_title", "views", "likes", "country"]]
print(f"High-performing videos (>=1M views & >50k likes): {len(high_perf):,}")

# (b) Top 20 records by engagement score
top_20_engagement = df.nlargest(20, "engagement_score")[["title", "channel_title", "category_name", "engagement_score", "country"]]
print("\n--- Top 5 Engagement Score Records ---")
display(top_20_engagement.head(5))

# (c) Music or Gaming videos above median views
overall_median_views = df["views"].median()
music_gaming_above_median = df.loc[
    df["category_name"].isin(["Music", "Gaming"]) & (df["views"] > overall_median_views),
    ["title", "category_name", "views", "country"]
]
print(f"\nMusic/Gaming videos above overall median views ({overall_median_views:,.0f}): {len(music_gaming_above_median):,}")

High-performing videos (>=1M views & >50k likes): 6,718

--- Top 5 Engagement Score Records ---


,title,channel_title,category_name,engagement_score,country
26526,Lena Ortiz - Monsoon (Live),FinalSpeech81,Music,0.953805,GB
26325,Lena Ortiz - Monsoon (Live),FinalSpeech81,Music,0.758036,GB
30954,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,Music,0.690224,GB
24108,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,Music,0.648929,GB
26123,Lena Ortiz - Monsoon (Live),FinalSpeech81,Music,0.624682,GB



Music/Gaming videos above overall median views (318,389): 9,126


In [11]:
# 1. Parse dates properly
df['trending_datetime'] = pd.to_datetime(df['trending_date'], format='%y.%d.%m', errors='coerce')

# Parse publish_time and strip timezone info to match trending_datetime (making both tz-naive)
df['publish_datetime'] = pd.to_datetime(df['publish_time'], errors='coerce').dt.tz_localize(None)

# 2. Days to trend (now both are tz-naive, subtraction will work smoothly)
df['days_to_trend'] = (df['trending_datetime'] - df['publish_datetime']).dt.days

# 3. Trending Lifespan: Number of distinct trending dates per video per country
lifespan_df = df.groupby(['country', 'video_id', 'title', 'category_name'])['trending_date'].nunique().reset_index()
lifespan_df.rename(columns={'trending_date': 'trending_lifespan_days'}, inplace=True)

print("--- Top 10 Longest-Lasting Videos on Trending List ---")
display(lifespan_df.nlargest(10, 'trending_lifespan_days')[['title', 'category_name', 'country', 'trending_lifespan_days']])

print("\n--- Average Trending Lifespan by Category ---")
avg_lifespan_cat = lifespan_df.groupby('category_name')['trending_lifespan_days'].mean().sort_values(ascending=False)
display(avg_lifespan_cat.head(10))

--- Top 10 Longest-Lasting Videos on Trending List ---


,title,category_name,country,trending_lifespan_days
950,Official Football Update,Entertainment,GB,25
1019,KAIRO - FESTIVAL (LIVE),Music,GB,25
1232,Kairo - Phones (Lyric Video)!,Music,GB,25
1432,Sunny Park - Cats (Live) MUST,Music,GB,25
1611,Anya Lux - Paris (Lyric Video),Music,GB,25
1630,MIRA SOLIS - WINTER (LYRIC VIDEO)?,Music,GB,25
1846,Lena Ortiz - Monsoon (Live),Music,GB,25
2343,Nova Kane - Robots (Official Video),Music,GB,25
2668,Final Cars Speech WATCH,Film & Animation,GB,25
2687,Sunny Park - Robots (Live),Music,GB,25



--- Average Trending Lifespan by Category ---


,trending_lifespan_days
category_name,
Music,5.333678
Entertainment,3.539083
Comedy,2.791774
People & Blogs,2.786813
Film & Animation,2.702353
Gaming,2.643959
Education,2.440653
Howto & Style,2.404480
Autos & Vehicles,2.369412


In [12]:
# 1. Combining per-country DataFrames using concat() (already done via df_raw, but let's confirm stack)
# df contains all combined records across countries

# 2. Per-video summary table joined with per-channel summary table using merge()
channel_summary = df.groupby('channel_title').agg(
    total_trending_videos=('video_id', 'count'),
    total_views_sum=('views', 'sum')
).reset_index()

video_summary = df[['video_id', 'title', 'channel_title', 'category_name', 'country']].drop_duplicates(subset=['video_id'])

# Merge channel stats back to video summary
combined_channel_analysis = pd.merge(
    video_summary,
    channel_summary,
    on='channel_title',
    how='inner'
)

print(f"Combined Channel & Video Summary Shape: {combined_channel_analysis.shape}")
print("\n--- Top Channels by Total Trending Views ---")
display(channel_summary.sort_values(by='total_views_sum', ascending=False).head(5))

Combined Channel & Video Summary Shape: (14699, 7)

--- Top Channels by Total Trending Views ---


,channel_title,total_trending_videos,total_views_sum
290,QuietChallenge75,2451,5669763684
432,WildUnboxing39,394,3019916986
132,FinalSpeech81,187,2362212889
281,PerfectTutorial78,538,2286945332
278,PerfectTrailer65,341,2253048658
